In [ ]:
# Node.js Execution Troubleshooting in PowerShell
This notebook helps verify Node.js availability in PowerShell and shows a reliable way to execute scripts and inspect `tmp/final_wa_outputs.log`.
</VSCode.Cell>
<VSCode.Cell language="python">
import os
from pathlib import Path
root = Path(r"c:\Users\TSC-AKA\Videos\MARKETING\BOTAI\system_wa")
print("Workspace root:", root)
print("Exists:", root.exists())
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 1. Install and Verify Node.js in PowerShell
</VSCode.Cell>
<VSCode.Cell language="python">
import subprocess
cmd = ["node", "--version"]
print("Command:", cmd)
try:
    completed = subprocess.run(cmd, cwd=root, capture_output=True, text=True, check=True)
    print("Node version:", completed.stdout.strip())
except subprocess.CalledProcessError as e:
    print("Node execution failed:")
    print(e.stdout)
    print(e.stderr)
except FileNotFoundError as e:
    print("Node executable not found:", e)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 2. Run a Node.js Script File
</VSCode.Cell>
<VSCode.Cell language="python">
script_path = root / "tmp" / "simulate_requests.js"
script_content = r'''
const fs = require('fs');
const path = require('path');
const fetch = global.fetch || require('node-fetch');
const base = 'http://127.0.0.1:4001';
const queries = [
  'Apa itu TI?',
  'Apa yang dipelajari di TI?',
  'Prospek kerja TI?',
  'Biaya kuliah TI?',
  'Akreditasi TI?',
  'Saya ingin daftar TI'
];
const logPath = path.join(__dirname, 'tmp', 'final_wa_outputs.log');
async function main() {
  try { fs.rmSync(logPath); } catch (e) {}
  for (const text of queries) {
    const res = await fetch(`${base}/_simulate`, {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ chatId: 'test-chat', text })
    });
    const body = await res.text();
    console.log('SIMULATE', text, '=>', res.status, body);
    await new Promise(r => setTimeout(r, 1200));
  }
  await new Promise(r => setTimeout(r, 1500));
  const content = fs.existsSync(logPath) ? fs.readFileSync(logPath, 'utf8') : '';
  console.log('===LOG START===');
  console.log(content);
  console.log('===LOG END===');
}
main().catch(err => {
  console.error(err);
  process.exit(1);
});
'''
script_path.write_text(script_content, encoding='utf-8')
print('Wrote', script_path)
print(script_content[:400], '...')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 3. Use PowerShell Here-Strings for Multiline JavaScript
This section demonstrates how to avoid direct JS typing in PowerShell by writing a script file and executing it.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. Send HTTP Requests from Node.js
</VSCode.Cell>
<VSCode.Cell language="python">
cmd = ["node", str(script_path)]
print("Running:", cmd)
completed = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
print("Return code:", completed.returncode)
print("STDOUT:\n", completed.stdout)
print("STDERR:\n", completed.stderr)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 5. Read and Log Output Files from Node.js
</VSCode.Cell>
<VSCode.Cell language="python">
log_path = root / "tmp" / "final_wa_outputs.log"
print("Log exists:", log_path.exists())
if log_path.exists():
    print(log_path.read_text(encoding='utf-8'))
else:
    print('No log file found.')
